# Proyecto 1 – Saber 11  
## Tarea 2: Selección, limpieza y alistamiento (Ingeniería de Datos)

Este notebook realiza la **limpieza completa** y además filtra **Bogotá** de forma robusta (sin importar mayúsculas/minúsculas, tildes o comillas en el texto).

**Entrada:** `proyectoSaber11.csv` (archivo grande con 51 columnas)  
**Salida principal:** `saber11_bogota_limpio.csv` (listo para análisis)  
**Salida para Excel:** `saber11_bogota_limpio_excel.csv` (separador `;`)


## 0) Configuración

- Pon este notebook en la misma carpeta donde está `proyectoSaber11.csv`.
- Si el archivo tiene otro nombre o está en otra ruta, cambia `INPUT_FILE`.


In [27]:
import pandas as pd
import numpy as np
import re

INPUT_FILE = "proyecto_Saber11.csv"
OUTPUT_MAIN = "saber11_bogota_limpio.csv"
OUTPUT_EXCEL = "saber11_bogota_limpio_excel.csv"


## 1) Carga del dataset

Usamos `low_memory=False` para evitar inferencias inconsistentes en archivos grandes. Ya empezamos a revisar el tamaño de la muestra y como se ven los datos que se extrajeron previamente.

In [28]:
df = pd.read_csv(INPUT_FILE, low_memory=False)
print("Shape original (filas, columnas):", df.shape)
df.head(10)

Shape original (filas, columnas): (1131725, 51)


,periodo,estu_tipodocumento,estu_consecutivo,cole_area_ubicacion,cole_bilingue,cole_calendario,cole_caracter,cole_cod_dane_establecimiento,cole_cod_dane_sede,cole_cod_depto_ubicacion,...,fami_tienecomputador,fami_tieneinternet,fami_tienelavadora,desemp_ingles,punt_ingles,punt_matematicas,punt_sociales_ciudadanas,punt_c_naturales,punt_lectura_critica,punt_global
0,20112,PV,SB11201120292808,URBANO,NaN,A,ACADÉMICO,3.110011e+11,311001107254,11,...,NaN,NaN,NaN,A-,36.00,27.00,NaN,NaN,NaN,NaN
1,20194,TI,SB11201940399505,URBANO,NaN,A,ACADÉMICO,1.110011e+11,111001096555,11,...,Si,Si,Si,A2,62.00,58.00,46.0,59.0,57.0,278.0
2,20194,TI,SB11201940399505,URBANO,NaN,A,ACADÉMICO,1.110011e+11,111001096555,11,...,Si,Si,Si,A2,62.00,58.00,46.0,59.0,57.0,278.0
3,20152,TI,SB11201520544825,URBANO,N,A,ACADÉMICO,3.110011e+11,311001106673,11,...,No,Si,Si,A1,54.00,50.00,60.0,54.0,50.0,268.0
4,20172,TI,SB11201720162770,URBANO,NaN,A,ACADÉMICO,1.110011e+11,111001098906,11,...,No,Si,No,A2,60.00,64.00,69.0,61.0,70.0,328.0
5,20102,CC,SB11201020502548,URBANO,N,A,ACADÉMICO,1.110010e+11,111001013170,11,...,No,No,No,A-,47.58,50.82,NaN,NaN,NaN,NaN
6,20142,TI,SB11201420094691,URBANO,N,A,ACADÉMICO,1.110010e+11,111001001121,11,...,Si,Si,Si,A2,59.00,69.00,65.0,66.0,67.0,331.0
7,20112,CC,SB11201120281539,URBANO,NaN,A,ACADÉMICO,1.110010e+11,111001015814,11,...,Si,Si,No,A-,41.00,45.00,NaN,NaN,NaN,NaN
8,20162,TI,SB11201620342593,URBANO,N,A,ACADÉMICO,1.110010e+11,111001014109,11,...,Si,Si,No,A2,65.00,63.00,58.0,62.0,62.0,306.0
9,20112,CC,SB11201120493992,URBANO,N,A,ACADÉMICO,3.110010e+11,311001031584,11,...,No,No,Si,A-,41.00,51.00,NaN,NaN,NaN,NaN


## 2) Exploración de datos

Comenzamos revisando el tipo de dato, los valores que se manejan en cada columna.

In [29]:
# Mostrar tipos de datos de todas las columnas
print("Tipos de datos por columna:")
print(df.dtypes)
print("\n" + "="*50)
print("\nResumen de tipos:")
print(df.dtypes.value_counts())

Tipos de datos por columna:
periodo                            int64
estu_tipodocumento                object
estu_consecutivo                  object
cole_area_ubicacion               object
cole_bilingue                     object
cole_calendario                   object
cole_caracter                     object
cole_cod_dane_establecimiento    float64
cole_cod_dane_sede                 int64
cole_cod_depto_ubicacion           int64
cole_cod_mcpio_ubicacion           int64
cole_codigo_icfes                float64
cole_depto_ubicacion              object
cole_genero                       object
cole_jornada                      object
cole_mcpio_ubicacion              object
cole_naturaleza                   object
cole_nombre_establecimiento       object
cole_nombre_sede                  object
cole_sede_principal               object
estu_cod_depto_presentacion      float64
estu_cod_mcpio_presentacion      float64
estu_cod_reside_depto            float64
estu_cod_reside_mcpio        

In [30]:
# valores únicos por columna (muestra)
for col in df.columns:
    uniques = df[col].dropna().unique()
    print(f"\n{col} (únicos: {len(uniques)})")
    print(uniques[:20])


periodo (únicos: 23)
[20112 20194 20152 20172 20102 20142 20162 20132 20122 20141 20191 20181
 20151 20201 20101 20121 20111 20131 20171 20211]

estu_tipodocumento (únicos: 15)
['PV' 'TI' 'CC' 'CR' 'PC' 'CE' 'PEP' 'NES' 'PE' 'V' 'RC' 'PPT' 'NUIP'
 'NIP' 'NUI']

estu_consecutivo (únicos: 961785)
['SB11201120292808' 'SB11201940399505' 'SB11201520544825'
 'SB11201720162770' 'SB11201020502548' 'SB11201420094691'
 'SB11201120281539' 'SB11201620342593' 'SB11201120493992'
 'SB11201940060190' 'SB11201940359696' 'SB11201120362539'
 'SB11201320265191' 'SB11201120340129' 'SB11201420339093'
 'SB11201940120759' 'SB11201220357952' 'SB11201220336132'
 'SB11201420421620' 'SB11201940498720']

cole_area_ubicacion (únicos: 2)
['URBANO' 'RURAL']

cole_bilingue (únicos: 2)
['N' 'S']

cole_calendario (únicos: 3)
['A' 'OTRO' 'B']

cole_caracter (únicos: 4)
['ACADÉMICO' 'TÉCNICO/ACADÉMICO' 'TÉCNICO' 'NO APLICA']

cole_cod_dane_establecimiento (únicos: 1396)
[3.11001107e+11 1.11001097e+11 3.11001107e+11 1.110

## 2) Normalización de texto para detectar Bogotá (robusto)

Vamos a construir una versión normalizada de los campos de ubicación:

1. Convertimos a texto
2. Quitamos comillas dobles
3. Quitamos tildes (ÁÉÍÓÚ → AEIOU)
4. Pasamos a minúsculas
5. Quitamos espacios extra

Luego filtramos todas las filas donde el **departamento** o el **municipio** contengan la palabra `bogota`.

In [31]:
def normalize_text(s: pd.Series) -> pd.Series:
    s = s.astype("string")
    s = s.str.replace('"', '', regex=False)  # quita comillas dobles
    # quita tildes solo para vocales (suficiente para Bogotá)
    s = s.str.translate(str.maketrans("ÁÉÍÓÚáéíóú", "AEIOUaeiou"))
    s = s.str.strip().str.lower()
    # normaliza espacios múltiples
    s = s.str.replace(r"\s+", " ", regex=True)
    return s

# Columnas de ubicación (colegio)
for col in ["cole_depto_ubicacion", "cole_mcpio_ubicacion"]:
    if col in df.columns:
        df[col + "_norm"] = normalize_text(df[col])
    else:
        print(f"No existe la columna {col} en este archivo.")

# Filtro Bogotá (por depto o por municipio)
mask_bogota = False
if "cole_depto_ubicacion_norm" in df.columns:
    mask_bogota = mask_bogota | df["cole_depto_ubicacion_norm"].str.contains("bogota", na=False)
if "cole_mcpio_ubicacion_norm" in df.columns:
    mask_bogota = mask_bogota | df["cole_mcpio_ubicacion_norm"].str.contains("bogota", na=False)

df_bog = df.loc[mask_bogota].copy()
print("Shape después de filtrar Bogotá:", df_bog.shape)

# Verifica valores originales más comunes (rápido)
if "cole_depto_ubicacion" in df_bog.columns:
    display(df_bog["cole_depto_ubicacion"].value_counts().head(10))
if "cole_mcpio_ubicacion" in df_bog.columns:
    display(df_bog["cole_mcpio_ubicacion"].value_counts().head(10))

Shape después de filtrar Bogotá: (1131725, 53)


cole_depto_ubicacion
BOGOTA    690385
BOGOTÁ    441340
Name: count, dtype: int64

cole_mcpio_ubicacion
BOGOTÁ D.C.     1039161
BOGOTÁ, D.C.      92564
Name: count, dtype: int64

## 3) Selección de columnas relevantes (alineado con las preguntas de negocio)

Para tus preguntas (estrato/hogar, ubicación, tipo de colegio), estas columnas son las más útiles.


In [32]:
keep_cols = [
    # contexto
    "periodo",
    # ubicación del colegio
    "cole_area_ubicacion", "cole_depto_ubicacion", "cole_mcpio_ubicacion",
    # características del colegio
    "cole_naturaleza", "cole_calendario", "cole_jornada", "cole_caracter",
    "cole_bilingue", "cole_genero",
    # hogar / estrato
    "fami_estratovivienda", "fami_personashogar", "fami_cuartoshogar",
    "fami_tieneinternet", "fami_tienecomputador",
    # puntajes
    "punt_global", "punt_matematicas", "punt_lectura_critica",
    "punt_c_naturales", "punt_sociales_ciudadanas", "punt_ingles",
    "desemp_ingles",
    # (opcional) identificador del estudiante si quieren deduplicar a nivel estudiante
    "estu_consecutivo"
]

existing = [c for c in keep_cols if c in df_bog.columns]
missing = [c for c in keep_cols if c not in df_bog.columns]

print("Columnas seleccionadas:", len(existing))
if missing:
    print("Columnas no encontradas (se omiten):", missing)

df_sel = df_bog[existing].copy()
df_sel.head(3)

Columnas seleccionadas: 23


,periodo,cole_area_ubicacion,cole_depto_ubicacion,cole_mcpio_ubicacion,cole_naturaleza,cole_calendario,cole_jornada,cole_caracter,cole_bilingue,cole_genero,...,fami_tieneinternet,fami_tienecomputador,punt_global,punt_matematicas,punt_lectura_critica,punt_c_naturales,punt_sociales_ciudadanas,punt_ingles,desemp_ingles,estu_consecutivo
0,20112,URBANO,BOGOTA,BOGOTÁ D.C.,NO OFICIAL,A,NOCHE,ACADÉMICO,NaN,MIXTO,...,NaN,NaN,NaN,27.0,NaN,NaN,NaN,36.0,A-,SB11201120292808
1,20194,URBANO,BOGOTÁ,BOGOTÁ D.C.,OFICIAL,A,TARDE,ACADÉMICO,NaN,MIXTO,...,Si,Si,278.0,58.0,57.0,59.0,46.0,62.0,A2,SB11201940399505
2,20194,URBANO,BOGOTÁ,BOGOTÁ D.C.,OFICIAL,A,TARDE,ACADÉMICO,NaN,MIXTO,...,Si,Si,278.0,58.0,57.0,59.0,46.0,62.0,A2,SB11201940399505


## 4) Limpieza de texto general (quitar comillas y espacios)

Aplicamos esto a todas las columnas tipo texto del subconjunto seleccionado.

In [33]:
obj_cols = df_sel.select_dtypes(include="object").columns
for c in obj_cols:
    df_sel[c] = (
        df_sel[c]
        .astype("string")
        .str.strip()
        .str.replace('"', "", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.translate(str.maketrans("ÁÉÍÓÚáéíóú", "AEIOUaeiou"))
        .str.replace(",", "", regex=False)
        .str.lower()
        .str.strip()
    )

print("Ejemplo depto (limpio):", df_sel["cole_depto_ubicacion"].dropna().unique()[:10] if "cole_depto_ubicacion" in df_sel.columns else "N/A")


Ejemplo depto (limpio): <StringArray>
['bogota']
Length: 1, dtype: string


## 5) Normalización SI/NO (internet y computador)

- Solo dejamos `SI` o `NO`
- Cualquier otro valor → `NA`

In [34]:
def normalizar_si_no(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().upper()
    if s in ["SI", "SÍ"]:
        return "SI"
    if s == "NO":
        return "NO"
    return pd.NA

for col in ["fami_tieneinternet", "fami_tienecomputador"]:
    if col in df_sel.columns:
        df_sel[col] = df_sel[col].apply(normalizar_si_no)

for col in ["fami_tieneinternet", "fami_tienecomputador"]:
    if col in df_sel.columns:
        print("\nFrecuencias:", col)
        display(df_sel[col].value_counts(dropna=False).head(10))



Frecuencias: fami_tieneinternet


fami_tieneinternet
SI      876571
NO      231253
<NA>     23901
Name: count, dtype: int64


Frecuencias: fami_tienecomputador


fami_tienecomputador
SI      898130
NO      214459
<NA>     19136
Name: count, dtype: int64

## 6) Feature engineering: estrato numérico

Creamos:
- `estrato_num` (1–6)
- `estrato_sin` (True si dice SIN ESTRATO)

In [35]:
if "fami_estratovivienda" in df_sel.columns:
    estr = df_sel["fami_estratovivienda"].astype("string").str.upper().str.strip()
    df_sel["estrato_num"] = estr.str.extract(r"ESTRATO\s+([1-6])", expand=False).astype("Float64")
    display(df_sel[["fami_estratovivienda","estrato_num"]].head(10))
    display(df_sel["estrato_num"].value_counts(dropna=False).sort_index())


,fami_estratovivienda,estrato_num
0,estrato 2,2.0
1,estrato 1,1.0
2,estrato 1,1.0
3,estrato 2,2.0
4,estrato 2,2.0
5,estrato 2,2.0
6,estrato 3,3.0
7,estrato 2,2.0
8,estrato 1,1.0
9,estrato 3,3.0


estrato_num
1.0     118579
2.0     483614
3.0     374309
4.0      81272
5.0      26491
6.0      18159
<NA>     29301
Name: count, dtype: Int64

## 7) Conversión a numérico de puntajes y conteos

Convertimos a numérico:
- puntajes
- número de personas y cuartos
- periodo

In [36]:
num_cols = [
    "periodo",
    "punt_global","punt_matematicas","punt_lectura_critica",
    "punt_c_naturales","punt_sociales_ciudadanas","punt_ingles",
    "fami_personashogar","fami_cuartoshogar"
]
for c in num_cols:
    if c in df_sel.columns:
        df_sel[c] = pd.to_numeric(df_sel[c], errors="coerce")

df_sel[[c for c in ["punt_global","punt_matematicas","punt_lectura_critica","punt_ingles"] if c in df_sel.columns]].describe()


,punt_global,punt_matematicas,punt_lectura_critica,punt_ingles
count,712661.000000,1.131725e+06,712661.000000,1.131671e+06
mean,272.171814,5.290127e+01,55.648569,5.327168e+01
std,49.131839,1.192393e+01,9.855946,1.372799e+01
min,0.000000,0.000000e+00,0.000000,-1.000000e+00
25%,237.000000,4.500000e+01,49.000000,4.300000e+01
50%,270.000000,5.200000e+01,56.000000,5.100000e+01
75%,306.000000,6.051000e+01,63.000000,6.000000e+01
max,494.000000,1.270000e+02,100.000000,1.172900e+02


In [37]:
# mostrar df_sel.head(10)
print("Tamaño del DataFrame final:", df_sel.shape)
print("\nPorcentaje de missing values:")
print((df_sel.isnull().sum() / len(df_sel) * 100).round(2))

Tamaño del DataFrame final: (1131725, 24)

Porcentaje de missing values:
periodo                       0.00
cole_area_ubicacion           0.16
cole_depto_ubicacion          0.00
cole_mcpio_ubicacion          0.00
cole_naturaleza               0.00
cole_calendario               0.03
cole_jornada                  0.00
cole_caracter                 1.96
cole_bilingue                13.39
cole_genero                   0.00
fami_estratovivienda          2.34
fami_personashogar          100.00
fami_cuartoshogar           100.00
fami_tieneinternet            2.11
fami_tienecomputador          1.69
punt_global                  37.03
punt_matematicas              0.00
punt_lectura_critica         37.03
punt_c_naturales             37.03
punt_sociales_ciudadanas     37.03
punt_ingles                   0.00
desemp_ingles                 0.03
estu_consecutivo              0.00
estrato_num                   2.59
dtype: float64


## 8) Reglas mínimas de calidad

- Eliminamos filas sin `punt_global`.
- Filtramos puntajes fuera de rango (0–500).
- Eliminamos duplicados exactos.
- Quitamos las columnas de personas en hogar y cuantos hogar.
- Para el caso de las informaciones que son texto y son bajos los porcentajes como menos del 5% se ingresará valores como no registra.
- Para el caso del estrato socioeconomico al ser bajo vamos a quitar esa información porque puede afectar nuestro analisis no contar con esos datos.
- Todas las columnas que cuenten con menos del 1% se eliminará la fila. 


In [38]:
before = df_sel.shape[0]

if "punt_global" in df_sel.columns:
    df_sel = df_sel.dropna(subset=["punt_global"])
    df_sel = df_sel[(df_sel["punt_global"] >= 0) & (df_sel["punt_global"] <= 500)]

if "estrato_num" in df_sel.columns:
    df_sel = df_sel.dropna(subset=["estrato_num"])
    df_sel = df_sel[((df_sel["estrato_num"] >= 1) & (df_sel["estrato_num"] <= 6))]

if "cole_area_ubicacion" in df_sel.columns:
    df_sel = df_sel.dropna(subset=["cole_area_ubicacion"])

if "cole_calendario" in df_sel.columns:
    df_sel = df_sel.dropna(subset=["cole_calendario"])

if "desemp_ingles" in df_sel.columns:   
    df_sel = df_sel.dropna(subset=["desemp_ingles"])

# Duplicados exactos
df_sel = df_sel.drop_duplicates()

# Quitar columnas de Fami_personashogar fami_cuartoshogar y fami_estratovivienda

df_sel = df_sel.drop(columns=[c for c in ["fami_personashogar", "fami_cuartoshogar","fami_estratovivienda"] if c in df_sel.columns], errors="ignore")

#nuevos valores a columnas de fami_tieneinternet, fami_tienecomputador cole_caracter cole_area_ubicacion

df_sel["fami_tieneinternet"] = df_sel["fami_tieneinternet"].fillna("SIN REGISTRO")
df_sel["fami_tienecomputador"] = df_sel["fami_tienecomputador"].fillna("SIN REGISTRO")
df_sel["cole_caracter"] = df_sel["cole_caracter"].fillna("SIN REGISTRO")
df_sel["cole_area_ubicacion"] = df_sel["cole_area_ubicacion"].fillna("SIN REGISTRO")
df_sel["cole_bilingue"] = df_sel["cole_bilingue"].fillna("SIN REGISTRO")



after = df_sel.shape[0]
print("Filas antes:", before)
print("Filas después:", after)
print("Removidas:", before - after)


Filas antes: 1131725
Filas después: 536615
Removidas: 595110


## 9) Guardado de salidas

- `OUTPUT_MAIN`: CSV estándar (coma) para trabajar en Python
- `OUTPUT_EXCEL`: CSV con `;` para que Excel lo abra en columnas correctamente

In [39]:
df_sel.to_csv(OUTPUT_MAIN, index=False, encoding="utf-8")
df_sel.to_csv(OUTPUT_EXCEL, index=False, sep=";", encoding="utf-8")

print("Guardado:", OUTPUT_MAIN)
print("Guardado (Excel):", OUTPUT_EXCEL)
print("Shape final:", df_sel.shape)


Guardado: saber11_bogota_limpio.csv
Guardado (Excel): saber11_bogota_limpio_excel.csv
Shape final: (536615, 21)
